<a href="https://colab.research.google.com/github/harjeet88/spark_examples/blob/master/llms/spark_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [120]:
!pip install -q pyspark faiss-cpu autofaiss sentence-transformers

In [121]:
import pandas as pd
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, col
from pyspark.sql.types import ArrayType, FloatType
from sentence_transformers import SentenceTransformer

In [122]:
# 1. Initialize Spark Session
spark = SparkSession.builder \
    .appName("EmbeddingGeneration") \
    .config("spark.driver.memory", "4g") \
    .getOrCreate()

In [123]:
# Sample data
data = [
    (1, "The quick brown fox jumps over the lazy dog."),
    (2, "A group of football players is running in the field."),
    (3, "The sky is blue and the sun is bright."),
    (4, "This is an example document about semantic search."),
    (5, "FAISS is a library for efficient similarity search.")
]

In [124]:
df = spark.createDataFrame(data, ["id", "text_content"])
df.show()

+---+--------------------+
| id|        text_content|
+---+--------------------+
|  1|The quick brown f...|
|  2|A group of footba...|
|  3|The sky is blue a...|
|  4|This is an exampl...|
|  5|FAISS is a librar...|
+---+--------------------+



In [125]:
# --- Distributed Embedding Generation using Pandas UDF ---
# The model chosen affects the dimension (e.g., all-MiniLM-L6-v2 produces 384-dim vectors)
EMBEDDING_MODEL_NAME = 'all-MiniLM-L6-v2'
EMBEDDING_DIMENSION = 384

In [129]:
# 3. Define the Pandas UDF for Embedding Generation
# The return type is an ArrayType of FloatType, matching the embedding vector.
@pandas_udf(ArrayType(FloatType()))
def generate_embeddings(content: pd.Series) -> pd.Series:
    """
    Pandas UDF to generate embeddings for a batch of text.
    This function runs on the Spark executor nodes.
    """
    try:
        # Load the model locally on the executor node.
        # This is efficient because it avoids re-downloading/loading for every row.
        model = SentenceTransformer(EMBEDDING_MODEL_NAME)

        # Encode the text batch
        # convert_to_numpy=True ensures the output is a NumPy array
        embeddings = model.encode(
            content.tolist(),
            convert_to_numpy=True,
            # Set batch_size appropriately for your worker memory
            batch_size=32
        )

        # Return embeddings as a Pandas Series of lists/arrays (PySpark requirement)
        return pd.Series(embeddings.tolist())
    except Exception as e:
        # Crucial for debugging issues on the executor
        print(f"Error during embedding generation: {e}")
        # Return a series of None for failed rows
        return pd.Series([None] * len(content))

In [130]:
# 4. Apply the UDF to the DataFrame
print(f"Applying UDF using model: {EMBEDDING_MODEL_NAME}...")
embeddings_df = df.withColumn(
    "embedding_vector",
    generate_embeddings(col("text_content"))
)

Applying UDF using model: all-MiniLM-L6-v2...


In [131]:
# 5. Display the Result and Schema
print("\n--- Result Schema ---")
embeddings_df.printSchema()

print("\n--- Sample Results (First 5 Rows) ---")
# The size of the array is the EMBEDDING_DIMENSION (384)
embeddings_df.show(5, truncate=30)


--- Result Schema ---
root
 |-- id: long (nullable = true)
 |-- text_content: string (nullable = true)
 |-- embedding_vector: array (nullable = true)
 |    |-- element: float (containsNull = true)


--- Sample Results (First 5 Rows) ---


PythonException: 
  An exception was thrown from the Python worker. Please see the stack trace below.
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pandas/__init__.py", line 37, in <module>
    from pandas._config import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/_config/__init__.py", line 20, in <module>
    from pandas._config import config
  File "/usr/local/lib/python3.12/dist-packages/pandas/_config/config.py", line 68, in <module>
    from pandas._typing import (
  File "/usr/local/lib/python3.12/dist-packages/pandas/_typing.py", line 198, in <module>
    np.random.Generator,
    ^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/__init__.py", line 337, in __getattr__
    public_symbols = globals().keys() | {'testing'}
        ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/numpy/random/__init__.py", line 180, in <module>
    from . import _pickle
  File "/usr/local/lib/python3.12/dist-p

In [ ]:
# 2. Save Embeddings to Parquet
# Parquet is excellent for columnar storage and is the recommended format for Spark.
# We save the ID and the embedding vector for later use.
PARQUET_PATH = "/tmp/embeddings_parquet"
embeddings_df.select("id", "embedding_vector").write.mode("overwrite").parquet(PARQUET_PATH)

print(f"Embeddings saved to Parquet at: {PARQUET_PATH}")

In [ ]:
import pandas as pd
import numpy as np
import faiss
import pyarrow.parquet as pq

# 3. Load Embeddings from Parquet
PARQUET_PATH = "/tmp/embeddings_parquet"
EMBEDDING_DIMENSION = 384 # Must match the model dimension

# Load the Parquet file using pyarrow/pandas
table = pq.read_table(PARQUET_PATH)
embeddings_pandas_df = table.to_pandas()

# Convert the list-of-lists column to a 2D NumPy array for FAISS
# Note: The 'embedding_vector' column is a list/array of floats
embeddings_matrix = np.stack(embeddings_pandas_df['embedding_vector'].to_numpy()).astype('float32')
ids = embeddings_pandas_df['id'].to_numpy().astype('int64')

print(f"Loaded embeddings shape: {embeddings_matrix.shape}")

In [ ]:
# 4. Build the FAISS Index
# IndexFlatL2 is a simple, exact search index using L2 (Euclidean) distance.
index_flat = faiss.IndexFlatL2(EMBEDDING_DIMENSION)

# To associate the vector index with our original document IDs, we use IndexIDMap
index = faiss.IndexIDMap(index_flat)

# Add the vectors and their IDs to the index
index.add_with_ids(embeddings_matrix, ids)

print(f"FAISS index built with {index.ntotal} vectors.")

# 5. Save the FAISS Index
FAISS_INDEX_PATH = "faiss_index.bin"
faiss.write_index(index, FAISS_INDEX_PATH)

print(f"FAISS index saved to: {FAISS_INDEX_PATH}")

# Task
Remove `numpy==1.23.5` from the `!pip install` command to resolve the wheel building issue.

## Remove explicit numpy version pin

### Subtask:
Remove `numpy==1.23.5` from the `!pip install` command to allow `pandas==1.5.3` to install a compatible `numpy` version, resolving the wheel building issue.


## Summary:

### Data Analysis Key Findings
*   The explicit `numpy==1.23.5` version pin was removed from the `!pip install` command. This action was taken to resolve a wheel building issue.
*   The removal of the specific `numpy` version is expected to allow `pandas==1.5.3` to install a compatible `numpy` version automatically, thereby resolving the dependency conflict.

### Insights or Next Steps
*   Verify that the wheel building issue is successfully resolved after the `numpy` version pin removal.
*   Confirm that all necessary packages, including `pandas==1.5.3`, can now be installed without conflicts.
